# 00 — First principles of contract analysis

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — Legal Tech |
| **Level** | Advanced |
| **Time** | ~30 min |
| **Prerequisites** | Python, basic NLP concepts |
| **Open in Colab** | `labs/03_contract_analyzer/00_first_principles.ipynb` |

> **Goal:** Understand *why* contracts are hard to analyze, *why* keyword matching
> fails, and *how* semantic understanding unlocks a $28 B+ legal-tech opportunity.

## Setup

In [ ]:
# ── Install dependencies ──
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0" "scikit-learn>=1.2.0" "matplotlib>=3.7.0"

import warnings
warnings.filterwarnings("ignore")
print("✓ Dependencies installed")

## 1 · What is a contract?

A contract is a **legally binding agreement** between two or more parties.
Every contract, regardless of length, contains these structural elements:

| Element | Purpose | Example |
|---|---|---|
| **Parties** | Who is bound | "Acme Corp" and "Client Inc" |
| **Obligations** | What each party must do | "Vendor shall deliver software by Q3" |
| **Conditions** | Triggers / prerequisites | "Upon receipt of payment…" |
| **Remedies** | What happens on breach | "Liquidated damages of $50K" |
| **Termination** | How the agreement ends | "Either party may terminate with 30 days notice" |

Below we annotate a **sample clause** and build a **clause taxonomy** that our
analyzer will use throughout the lab.

In [ ]:
# ── Annotate a sample contract structure ──
from typing import Dict, List

SAMPLE_CONTRACT_STRUCTURE: Dict[str, str] = {
    "parties": "This Agreement is between TechVendor Inc. ('Vendor') and GlobalCorp Ltd. ('Client').",
    "obligations": "Vendor shall deliver the Software Platform by March 31, 2025, "
                   "including all modules specified in Exhibit A.",
    "conditions": "Payment of the license fee is due within 30 days of each milestone delivery.",
    "remedies": "In the event of material breach, the non-breaching party may seek "
                "liquidated damages not to exceed $500,000.",
    "termination": "Either party may terminate this Agreement with 60 days written notice, "
                   "provided all outstanding obligations are settled.",
}

for element, text in SAMPLE_CONTRACT_STRUCTURE.items():
    print(f"[{element.upper():^15}] {text}")
    print()

# ── Clause taxonomy ──
CLAUSE_TAXONOMY: Dict[str, str] = {
    "indemnification":       "One party compensates the other for losses or damages",
    "limitation_of_liability": "Caps on the amount or types of damages recoverable",
    "IP_assignment":          "Transfer or licensing of intellectual property rights",
    "termination":            "Conditions under which the agreement may be ended",
    "confidentiality":        "Obligations to protect non-public information",
    "force_majeure":          "Excuse for non-performance due to extraordinary events",
    "non_compete":            "Restrictions on competitive activities post-contract",
    "payment_terms":          "Timing, method, and conditions of payment",
    "warranty":               "Guarantees about quality, performance, or fitness",
    "dispute_resolution":     "Mechanism for resolving disagreements (arbitration, litigation)",
}

print(f"Clause taxonomy — {len(CLAUSE_TAXONOMY)} types:")
for ctype, desc in CLAUSE_TAXONOMY.items():
    print(f"  • {ctype:28s} → {desc}")

## 2 · What makes a clause risky?

Risk in a contract clause comes from three main sources:

1. **Asymmetric obligations** — one party bears disproportionate burden
2. **Vague remedies** — undefined consequences leave exposure open-ended
3. **Unbounded liability** — no caps, no carve-outs, unlimited exposure

Below we compare **safe** vs **risky** versions of five common clause types
and build a **risk checklist** as structured data.

In [ ]:
# ── Safe vs risky clause pairs ──
from typing import NamedTuple

class ClausePair(NamedTuple):
    clause_type: str
    safe: str
    risky: str
    risk_reason: str

CLAUSE_PAIRS: List[ClausePair] = [
    ClausePair(
        "indemnification",
        safe="Each party shall indemnify the other for direct damages caused by its "
             "material breach, capped at the total fees paid in the prior 12 months.",
        risky="Vendor shall indemnify Client against any and all claims, losses, and "
              "expenses of any kind, without limitation, arising from or related to this Agreement.",
        risk_reason="Unbounded — no cap, covers 'any and all' including consequential damages",
    ),
    ClausePair(
        "limitation_of_liability",
        safe="Neither party's aggregate liability shall exceed the total amount paid "
             "under this Agreement in the 12 months preceding the claim.",
        risky="Client's liability under this Agreement shall not exceed $100. Vendor's "
              "liability is unlimited for all claims.",
        risk_reason="Asymmetric — Client capped at trivial amount while Vendor has unlimited exposure",
    ),
    ClausePair(
        "termination",
        safe="Either party may terminate with 60 days written notice. All prepaid fees "
             "for undelivered services shall be refunded.",
        risky="Client may terminate at any time without cause. Vendor may not terminate "
              "and must continue performance regardless of Client's payment status.",
        risk_reason="One-sided — only Client can exit; Vendor trapped even without payment",
    ),
    ClausePair(
        "IP_assignment",
        safe="Vendor retains all pre-existing IP. Work product created specifically for "
             "Client under this Agreement is assigned to Client upon full payment.",
        risky="All intellectual property conceived, developed, or delivered by Vendor "
              "during the term, including pre-existing IP, shall be the sole property of Client.",
        risk_reason="Overreach — captures Vendor's pre-existing IP, not just project deliverables",
    ),
    ClausePair(
        "confidentiality",
        safe="Each party shall keep Confidential Information secret for 3 years after "
             "termination, subject to customary exceptions (legal obligation, public domain).",
        risky="Vendor shall keep all information received from Client confidential in "
              "perpetuity. No exceptions. Client has no reciprocal obligation.",
        risk_reason="Perpetual, non-mutual, no carve-outs — practically unenforceable and one-sided",
    ),
]

for i, pair in enumerate(CLAUSE_PAIRS, 1):
    print(f"{'─'*80}")
    print(f"Pair {i}: {pair.clause_type.upper()}")
    print(f"  ✅ SAFE:  {pair.safe[:100]}…")
    print(f"  ❌ RISKY: {pair.risky[:100]}…")
    print(f"  ⚠  WHY:   {pair.risk_reason}")
print(f"{'─'*80}")

# ── Build risk checklist ──
RISK_CHECKLIST: List[Dict[str, str]] = [
    {"factor": "Unbounded liability",   "signal": "No cap on damages or indemnification",     "severity": "Critical"},
    {"factor": "Asymmetric obligations", "signal": "One party bears disproportionate burden",  "severity": "High"},
    {"factor": "Vague remedies",         "signal": "'Reasonable efforts' with no definition",  "severity": "High"},
    {"factor": "Perpetual obligations",  "signal": "No time limit on confidentiality/non-compete", "severity": "Medium"},
    {"factor": "IP overreach",           "signal": "Captures pre-existing or unrelated IP",    "severity": "Critical"},
    {"factor": "Unilateral termination", "signal": "Only one party can exit",                  "severity": "High"},
    {"factor": "Auto-renewal traps",     "signal": "Silent auto-renew with price escalation",  "severity": "Medium"},
    {"factor": "Broad force majeure",    "signal": "Excuses performance for minor disruptions","severity": "Low"},
]

print("\nRisk Checklist:")
print(f"{'Factor':<25} {'Signal':<50} {'Severity':<10}")
print("─" * 85)
for item in RISK_CHECKLIST:
    print(f"{item['factor']:<25} {item['signal']:<50} {item['severity']:<10}")

## 3 · Why keyword matching fails

Traditional contract-review tools rely on **keyword search** — scanning for terms
like *"indemnify"*, *"liability"*, *"terminate"*.  This works for obvious cases but
**misses semantically equivalent language** that avoids trigger words.

Below we build a keyword-based risk flagger, test it on 10 clauses (5 risky, 5 safe),
and measure its **recall** — how many truly risky clauses it catches.

In [ ]:
import re
from typing import Tuple

# ── Keyword-based risk flagger ──
RISK_KEYWORDS: List[str] = [
    "indemnify", "indemnification", "indemnities",
    "liability", "liable",
    "terminate", "termination",
    "damages", "penalty", "forfeit",
    "unlimited", "without limitation",
]

def keyword_risk_flag(clause: str) -> Tuple[bool, List[str]]:
    """Flag a clause as risky if it contains any risk keywords."""
    clause_lower = clause.lower()
    hits = [kw for kw in RISK_KEYWORDS if kw in clause_lower]
    return (len(hits) > 0, hits)

# ── Test clauses: 5 risky (labeled True), 5 safe (labeled False) ──
TEST_CLAUSES: List[Tuple[str, bool]] = [
    # Risky — uses keywords
    ("Vendor shall indemnify Client against all claims without limitation.", True),
    ("Total liability under this agreement shall not exceed $100.", True),
    # Risky — NO keywords (semantic equivalents)
    ("The provider assumes full financial responsibility for any adverse outcomes "
     "experienced by the recipient, regardless of cause or amount.", True),
    ("Should the recipient choose to discontinue the arrangement, the provider "
     "must continue all obligations and absorb associated costs.", True),
    ("All creative works, inventions, and innovations produced during engagement "
     "become the exclusive property of the recipient in perpetuity.", True),
    # Safe clauses
    ("Each party shall maintain reasonable data-security practices.", False),
    ("Notices shall be sent to the addresses listed in Exhibit B.", False),
    ("This agreement shall be governed by the laws of Delaware.", False),
    ("The parties agree to attempt mediation before litigation.", False),
    ("Vendor shall deliver monthly progress reports.", False),
]

print(f"{'Clause (truncated)':<70} {'Label':>6} {'Flagged':>8} {'Keywords'}")
print("─" * 120)

true_positives = 0
false_negatives = 0
false_positives = 0
true_negatives = 0

for clause_text, is_risky in TEST_CLAUSES:
    flagged, hits = keyword_risk_flag(clause_text)
    short = clause_text[:67] + "…" if len(clause_text) > 67 else clause_text
    status = "✓" if flagged == is_risky else "✗"
    print(f"{status} {short:<68} {str(is_risky):>6} {str(flagged):>8}  {hits}")

    if is_risky and flagged:
        true_positives += 1
    elif is_risky and not flagged:
        false_negatives += 1
    elif not is_risky and flagged:
        false_positives += 1
    else:
        true_negatives += 1

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) else 0

print(f"\n📊 Results:")
print(f"   True Positives:  {true_positives}   |   False Negatives: {false_negatives}")
print(f"   False Positives: {false_positives}   |   True Negatives:  {true_negatives}")
print(f"   Precision: {precision:.0%}   |   Recall: {recall:.0%}")
print(f"\n⚠  Keyword matching catches only {recall:.0%} of risky clauses — "
      f"it misses semantically risky language that avoids trigger words.")

## 4 · Semantic understanding of legal language

Unlike keyword matching, **embedding models** capture the *meaning* of text.
Clauses about the same concept cluster together even when they use
completely different words.

We embed all 10 test clauses with `sentence-transformers`, reduce to 2-D
with PCA, and visualize the result.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# ── Embed all clauses ──
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c[0] for c in TEST_CLAUSES]
labels = ["risky" if c[1] else "safe" for c in TEST_CLAUSES]
embeddings = model.encode(texts, show_progress_bar=False)

print(f"Embedding shape: {embeddings.shape}")
print(f"Clauses: {len(texts)}  |  Risky: {labels.count('risky')}  |  Safe: {labels.count('safe')}")

# ── PCA to 2-D ──
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)

# ── Visualize ──
fig, ax = plt.subplots(figsize=(9, 6))
colors = {"risky": "#e74c3c", "safe": "#2ecc71"}
for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, c=colors[labels[i]], s=120, edgecolors="white", linewidths=0.8, zorder=3)
    short_label = texts[i][:40] + "…"
    ax.annotate(short_label, (x, y), fontsize=7, alpha=0.8,
                xytext=(5, 5), textcoords="offset points")

ax.set_title("Clause embeddings — PCA projection", fontsize=13, fontweight="bold")
ax.set_xlabel(f"PC-1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC-2 ({pca.explained_variance_ratio_[1]:.1%} variance)")

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#e74c3c", label="Risky"), Patch(color="#2ecc71", label="Safe")],
          loc="best", framealpha=0.9)
plt.tight_layout()
plt.show()

print("\n✓ Risky clauses cluster together even when they avoid risk keywords.")
print("  This is why semantic understanding beats keyword matching.")

## 5 · The analysis pipeline

To turn semantic understanding into a **production system**, we decompose the
problem into discrete, testable stages:

```
Contract Text
    │
    ▼
┌──────────────────┐
│  1. Segmentation  │  Split document into individual clauses
└───────┬──────────┘
        ▼
┌──────────────────┐
│ 2. Classification │  Label each clause by type (indemnity, IP, etc.)
└───────┬──────────┘
        ▼
┌──────────────────┐
│  3. Assessment    │  Score risk for each clause (1–5 scale)
└───────┬──────────┘
        ▼
┌──────────────────┐
│  4. Comparison    │  Compare against standard/market clauses (RAG)
└───────┬──────────┘
        ▼
┌──────────────────┐
│ 5. Recommendation │  Generate negotiation talking points
└───────┬──────────┘
        ▼
┌──────────────────┐
│ 6. Report Gen     │  Produce executive summary + clause-by-clause detail
└──────────────────┘
```

Each stage **narrows the problem space** for the next.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class PipelineStage:
    """Represents one stage in the contract analysis pipeline."""
    name: str
    input_type: str
    output_type: str
    purpose: str
    reduces_problem_by: str

PIPELINE_STAGES: List[PipelineStage] = [
    PipelineStage("Segmentation",   "Raw contract text",  "List[Clause]",
                  "Split document into individual clauses",
                  "Turns unstructured doc into structured units"),
    PipelineStage("Classification", "List[Clause]",       "List[(Clause, Type)]",
                  "Label each clause by legal category",
                  "Enables type-specific risk rules"),
    PipelineStage("Assessment",     "List[(Clause, Type)]", "List[(Clause, Type, RiskScore)]",
                  "Score risk 1-5 for each clause",
                  "Prioritizes reviewer attention to highest-risk items"),
    PipelineStage("Comparison",     "Risky clauses",      "List[Deviation]",
                  "Compare against market-standard versions via RAG",
                  "Grounds risk in concrete deviations from norms"),
    PipelineStage("Recommendation", "List[Deviation]",    "List[NegotiationPoint]",
                  "Generate alternative language + talking points",
                  "Makes risk actionable for non-lawyers"),
    PipelineStage("Report",         "All stage outputs",  "StructuredReport",
                  "Executive summary + clause detail table",
                  "One deliverable replaces hours of manual review"),
]

print("Contract Analysis Pipeline")
print("=" * 90)
for i, stage in enumerate(PIPELINE_STAGES, 1):
    print(f"\n  Stage {i}: {stage.name}")
    print(f"    Input:    {stage.input_type}")
    print(f"    Output:   {stage.output_type}")
    print(f"    Purpose:  {stage.purpose}")
    print(f"    Reduces:  {stage.reduces_problem_by}")
print("\n" + "=" * 90)
print("Each stage converts ambiguity into structure, narrowing the problem for the next.")

## 6 · Market analysis

The legal-tech market is booming — and contract analysis sits at its center.

| Metric | Value |
|---|---|
| Global legal spend | $1 T+ / year |
| Legal-tech market (2024) | $28 B+ |
| Contract-review hourly rate | $300 – $600 / hr |
| Average contract review time | 2 – 4 hours |
| Harvey AI (case study) | $100 M+ ARR, Series C |

AI-powered contract analysis reduces cost by **80-90%** while increasing
consistency and coverage.

In [ ]:
# ── Cost comparison: manual vs AI-assisted contract review ──
from typing import Dict

SCENARIOS: List[Dict[str, object]] = [
    {"scenario": "Single NDA",           "pages": 5,   "manual_hrs": 1.0,  "ai_min": 2,   "hourly_rate": 400},
    {"scenario": "SaaS agreement",       "pages": 20,  "manual_hrs": 3.0,  "ai_min": 5,   "hourly_rate": 450},
    {"scenario": "Enterprise license",   "pages": 50,  "manual_hrs": 6.0,  "ai_min": 8,   "hourly_rate": 500},
    {"scenario": "M&A due diligence (100 contracts)", "pages": 2000, "manual_hrs": 200, "ai_min": 120, "hourly_rate": 550},
]

print(f"{'Scenario':<40} {'Pages':>6} {'Manual':>10} {'AI':>10} {'Savings':>10} {'Cost ↓':>8}")
print("─" * 90)

for s in SCENARIOS:
    manual_cost = s["manual_hrs"] * s["hourly_rate"]
    ai_cost = (s["ai_min"] / 60) * 50  # $50/hr AI cost estimate
    savings = manual_cost - ai_cost
    pct = savings / manual_cost * 100
    print(f"{s['scenario']:<40} {s['pages']:>6} "
          f"${manual_cost:>8,.0f} ${ai_cost:>8,.0f} ${savings:>8,.0f} {pct:>6.0f}%")

print("\n💡 Harvey AI case study:")
print("   • Founded 2022, $100M+ ARR by 2024")
print("   • Elite law firms (Allen & Overy) using it for contract review")
print("   • Demonstrates massive market pull for LLM-based legal analysis")

## Exercises

1. **Extend the taxonomy** — Add 3 more clause types to `CLAUSE_TAXONOMY` (e.g.,
   `governing_law`, `assignment`, `audit_rights`). For each, write a safe and risky
   example clause.  Verify the keyword flagger misses at least one risky version.

2. **Improve keyword matching** — Add 10 more keywords or regex patterns to
   `RISK_KEYWORDS`.  Re-run the evaluation.  Can you get recall above 80%?
   What happens to precision?

3. **Embedding experiment** — Pick 5 new clauses (mix of risky and safe).  Encode
   them and project onto the existing PCA space.  Do they land in the expected
   clusters?

## Key Takeaways

| # | Takeaway |
|---|---|
| 1 | Contracts have a predictable structure — parties, obligations, conditions, remedies, termination |
| 2 | Risk comes from asymmetry, vagueness, and unbounded exposure |
| 3 | Keyword matching catches obvious risk but misses semantic equivalents (low recall) |
| 4 | Embedding models capture meaning — risky clauses cluster regardless of wording |
| 5 | A staged pipeline (segment → classify → score → compare → recommend) makes the problem tractable |
| 6 | The market opportunity is massive: $28B+ legal tech, 80-90% cost reduction possible |

## What's Next

In **01 — Architecture** we design the full system: clause segmenter,
classifier, risk scorer, standard-clause library (RAG), and report generator.

## References

1. Harvey AI — *AI for Elite Law Firms*, https://www.harvey.ai
2. McKinsey & Company — *The State of AI in Legal*, 2024 report
3. Reimers, N. & Gurevych, I. — *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*, EMNLP 2019
4. Thomson Reuters — *2024 Legal Technology Survey Report*